# complaint-ai 모델 학습 (Gemma 4 E4B QLoRA)

학습 대상 (자세한 기준은 `model-training/data/label_guide.md`)
- **챗봇 모드 0** : 도구 선택(접수 `insertRequestT` / 조회 `selectComplaintT` / 도구 없음) + 인자 + 안내 문구
- **민원 분석** : 카테고리·요약·핵심어 JSON (`data/analysis_records.jsonl` 이 있을 때만)

순서: **0~3 준비 → 4 설정 → 5 데이터 → 6 모델 → 6-1 학습 전 평가 → 7 렌더링 확인 → 8 학습 → 9 학습 후 평가·비교 → 10 저장 → 11 GGUF 내보내기(Ollama 용 병합) / 11-B llama-server 용(베이스 + 어댑터)**

앱의 `dcm-back/.env` 에서 `LLM_BACKEND=ollama` 로 쓸 거면 **11번**, `LLM_BACKEND=llama_server` 로 쓸 거면 **11-B** 를 실행합니다. 둘 다 해도 됩니다.

**처음에는 4단계에서 `SMOKE = True` 로 끝까지 한 번 돌리세요.** 데이터 몇십 건·10스텝만 학습하고 GGUF·Modelfile 까지 만들어
전체 경로(학습 → 병합 → GGUF → Ollama 등록)가 되는지 20~40분 안에 확인합니다. 그다음 `SMOKE = False` 로 본 학습을 합니다.

GPU: **L4 이상 권장** (bfloat16 지원). T4 는 float16 이라 loss 가 nan 이 될 수 있습니다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 드라이브 연결
저장소 폴더(`dcm-back`, `model-training` 이 들어 있는 `doc-classifier-ml-main`)를 **내 드라이브 맨 위**에 올려 두세요.
다른 위치면 아래 `REPO` 를 고치세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/doc-classifier-ml-main'
%cd {REPO}
!ls
!ls model-training model-training/data

## 2. 설치 (5~10분) → 끝나면 3번으로 런타임 재시작

In [ ]:
!pip install -q -r model-training/requirements-train.txt
!pip uninstall -y -q torchao   # Colab 기본 torchao(낡은 버전)가 peft 병합 단계에서 ImportError 를 냄 - 이 프로젝트는 쓰지 않음
# 그래프 한글 글꼴 (없으면 지표 이름이 네모로 깨짐). 3번 재시작 뒤 matplotlib 이 새로 읽습니다
!apt-get -qq install -y fonts-nanum > /dev/null && rm -rf ~/.cache/matplotlib
print('설치 완료 - 아래 3번 셀로 런타임을 재시작하세요')

## 3. 런타임 재시작 ('세션이 다시 시작되었습니다' 는 정상)

In [ ]:
import os
os.kill(os.getpid(), 9)

## 4. 재시작 후 준비 + 학습 설정
**런타임이 끊기면 여기부터** 다시 실행하면 됩니다. (설치는 남아 있음)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO = '/content/drive/MyDrive/doc-classifier-ml-main'
%cd {REPO}

import os, sys, json, time, gc
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
sys.path[:0] = [f'{REPO}/model-training', f'{REPO}/dcm-back']

import torch, transformers, peft
import pandas as pd
import common, records, samples, trainer, evaluate, export, plots
from app import prompts

print('GPU          :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
print('bfloat16     :', trainer.supports_bf16())
print('transformers :', transformers.__version__, '| peft', peft.__version__)


# ===================== 학습 설정 =====================
ANALYSIS_PROMPT_STYLE = 'long'   # 민원 분석 프롬프트 길이: 'long'(규칙 전체 약 3,000자) / 'short'(카테고리 이름·형식만 약 300자)
                                 # short 는 규칙을 데이터로 익혀 학습·실행이 빨라짐. 앱 .env 의 ANALYSIS_PROMPT_STYLE 도 같은 값으로
os.environ['ANALYSIS_PROMPT_STYLE'] = ANALYSIS_PROMPT_STYLE
SMOKE = False                 # True: 전체 경로 점검 (소량·10스텝). 본 학습은 False
RUN_NAME = ''                # 비우면 v날짜_시각 자동
REPLY_RATIO = 0.5            # 도구 레코드 중 '도구 결과 뒤 안내 문장' 샘플도 만들 비율
# 학습 데이터는 정답 종류별로 같은 비율로 고릅니다. (records.pick_balanced - 500개든 1,000개든 전부든 비율이 같음)
#   챗봇 6종: 접수 / 접수(내용 없음) / 조회(대상 있음) / 조회(대상 없음) / 거절 / 문의 안내  -> 각 1/6
#   분석 7종: 카테고리                                                                      -> 각 1/7
TRAIN_LIMIT = 1000           # 학습에 쓸 챗봇 레코드 수 (예: 500, 1000). 0 이면 학습 분할 전체 건수만큼 (비율은 똑같이 맞춤)
MAX_REPEAT = 3               # 어떤 종류의 건수가 몫보다 적을 때 몇 번까지 반복해 채울지 (그래도 모자라면 다른 종류가 나눠 채움)
ANALYSIS_REPEAT = 2          # 분석 샘플 수 = 학습 분석 레코드 수 x 이 값. 분석 학습 레코드 280건 기준
                             # (TRAIN_LIMIT=1000, 분석 학습 485건 기준 분석 비중: 1 = 약 27% / 2 = 약 42%). 8번 표에서 analysis val loss 가 먼저 오르면 줄이세요
                             # 카테고리마다 같은 수가 되도록 고르므로, 적은 카테고리는 더 반복되고 많은 카테고리는 덜 반복됩니다
EVAL_LIMIT = 300             # 평가할 레코드 수 (0 이면 평가셋 전부 - 오래 걸림). 종류별 같은 수, 학습 전·후 같은 레코드로 평가
RUN_BASELINE = True          # 학습 전 평가 (같은 조건의 결과 파일이 있으면 다시 계산하지 않고 불러옴)
GGUF_OUTTYPE = 'q8_0'        # q8_0(약 8GB, 손실 거의 없음) / bf16(PC 에서 ollama create --quantize q4_K_M)
cfg = trainer.TrainConfig(
    learning_rate=1e-4, epochs=1.0, batch_size=2, grad_accum=4,
    lora_r=16, lora_alpha=32, lora_dropout=0.1,
    logging_steps=1,         # train loss 를 몇 스텝마다 기록·그래프 갱신할지 (1 = 매 스텝)
)
# ====================================================
if SMOKE:
    cfg.max_steps, cfg.evals_per_epoch, cfg.early_stopping_patience = 10, 2, 0
    EVAL_LIMIT = 20
RUN_NAME = RUN_NAME or ('smoke_' if SMOKE else 'v') + time.strftime('%m%d_%H%M')
print('RUN_NAME     :', RUN_NAME, '| SMOKE', SMOKE)
print('베이스 모델   :', common.BASE_MODEL, '| 프롬프트', prompts.prompt_version(), f'(분석 {prompts.analysis_prompt_style()})', '| 평가', evaluate.EVAL_VERSION)
PLOT_DIR = common.OUTPUT_DIR / 'plots'   # 그래프 PNG 저장 위치
plots.setup_korean_font()


def run_eval(model, label):
    # 평가셋(test)으로 지표 계산 - 학습 전·후 같은 함수·같은 레코드
    model.config.use_cache = True
    gen = evaluate.hf_generator(model, tokenizer)
    m = {'chat': evaluate.evaluate_chat(gen, data['test'][0], EVAL_LIMIT or None, reply_limit=20 if SMOKE else 100, label=label)}
    if data['test'][1]:
        m['analysis'] = evaluate.evaluate_analysis(gen, data['test'][1], EVAL_LIMIT or None, label=label)
    for task, mm in m.items():
        print(f'[{label} - {task}]'); display(pd.DataFrame(evaluate.summary_table(mm)))
    return m, gen


def _scores(m):
    return {t: {k: v for k, v in mm.items() if not k.startswith('_')} for t, mm in m.items()}

## 5. 데이터 검증·불러오기
`records.py` 가 label_guide.md 의 규칙으로 검사합니다. 오류가 있으면 여기서 멈춥니다.

학습·검증 데이터는 **정답 종류별로 같은 비율**로 고릅니다. (`records.pick_balanced`)
- 챗봇 6종(접수 / 접수(내용 없음) / 조회(대상 있음) / 조회(대상 없음) / 거절 / 문의 안내)이 각 1/6, 분석은 카테고리 7종이 각 1/7
- `TRAIN_LIMIT` 을 500, 1000 처럼 바꿔도 비율은 같습니다. 같은 건수면 항상 같은 레코드이고, 건수를 늘리면 앞서 고른 레코드에 더 붙습니다.
- 어떤 종류가 몫보다 적으면 `MAX_REPEAT` 번까지 반복해 채웁니다. 아래 표에서 종류별 건수가 같은지 확인하세요.

In [ ]:
if records.main([]) != 0:
    raise SystemExit('❌ 데이터 오류 - 위 목록을 고친 뒤 다시 실행하세요.')

chat = records.load_chat_records()
analysis = records.load_analysis_records() if common.ANALYSIS_RECORDS.exists() else []
split = lambda rows, name: [r for r in rows if r['split'] == name]
data = {name: (split(chat, name), split(analysis, name)) for name in ('train', 'val', 'test')}

# 정답 종류별로 같은 비율로 고름 (같은 건수면 항상 같은 레코드). 평가셋(test)은 평가할 때 EVAL_LIMIT 으로 고름
pick_c = lambda rows, n, rep=MAX_REPEAT: records.pick_balanced(rows, n, records.chat_kind, max_repeat=rep)
pick_a = lambda rows, n, rep=ANALYSIS_REPEAT + 1: records.pick_balanced(rows, n, records.analysis_kind, max_repeat=rep)
if SMOKE:
    data['train'] = (pick_c(data['train'][0], 60), pick_a(data['train'][1], 14))
    data['val'] = (pick_c(data['val'][0], 18, 1), pick_a(data['val'][1], 7, 1))
else:
    data['train'] = (pick_c(data['train'][0], TRAIN_LIMIT or None),
                     pick_a(data['train'][1], len(data['train'][1]) * ANALYSIS_REPEAT))
    # 검증은 loss 확인용 - 반복 없이 종류별 같은 수. 챗봇은 240건(6종 x 40)으로 줄여 학습 시간을 아낌
    data['val'] = (pick_c(data['val'][0], 240 if TRAIN_LIMIT else None, 1), pick_a(data['val'][1], None, 1))
display(pd.DataFrame([{'분할': k, '챗봇': len(v[0]), '분석': len(v[1])} for k, v in data.items()]))
kinds = lambda rows: {records.CHAT_KIND_NAMES.get(k, k): n for k, n in records.kind_counts(rows, records.chat_kind).items()}
for k in ('train', 'val'):
    print(f'[{k}] 챗봇 정답 종류:', kinds(data[k][0]))
    print(f'[{k}] 분석 카테고리 :', records.kind_counts(data[k][1], records.analysis_kind))
if not analysis:
    print('analysis_records.jsonl 이 없어 챗봇만 학습합니다.')

## 6. 모델 올리기 (4bit) - 첫 실행은 다운로드 5~15분

In [ ]:
tokenizer, model = trainer.load_base_model(common.BASE_MODEL, four_bit=True)
print(type(model).__name__, '| GPU 메모리', round(torch.cuda.memory_allocated() / 1e9, 1), 'GB')

## 6-1. 학습 전 평가 (같은 Gemma 4 E4B, LoRA 없이)
학습 후 평가(9번)와 **같은 평가 레코드**로 계산해, 9번에서 학습 전·후를 나란히 비교합니다.
결과는 `outputs/eval_before_<프롬프트 버전>_<평가 버전>_n<건수>.json` 에 저장하고, 같은 조건이면 다음부터는 불러옵니다.
(평가셋·프롬프트·평가 방식(`evaluate.EVAL_VERSION`)을 바꾸면 파일 이름이 달라져 다시 계산합니다. 강제로 다시 재려면 파일을 지우세요)

**진행 표시** - 단계마다 한 줄이 제자리에서 갱신됩니다. 전체 수가 정해져 있어 남은 시간이 나옵니다.
```
[학습 전] 챗봇 도구 호출  120/300 (40%) | 경과 6.1분 | 남은 시간 약 9.2분
[학습 전] 챗봇 안내 문구  100/100 (100%) | 경과 3.4분
[학습 전] 민원 분석  12/29 (41%) | 경과 1.8분 | 남은 시간 약 2.6분
```
한 문장마다 생성이 있어 가장 오래 걸리는 셀입니다. `EVAL_LIMIT` 으로 챗봇 평가 건수를 줄일 수 있습니다.

In [ ]:
baseline_metrics = {}
if RUN_BASELINE:
    tag = f"{prompts.prompt_version()}_{evaluate.EVAL_VERSION}_n{EVAL_LIMIT or 'all'}_{len(data['test'][0])}c{len(data['test'][1])}a" + ('_smoke' if SMOKE else '')
    baseline_file = common.OUTPUT_DIR / f'eval_before_{tag}.json'
    if baseline_file.exists():
        baseline_metrics = json.loads(baseline_file.read_text(encoding='utf-8'))
        print('저장된 학습 전 평가를 불러왔습니다:', baseline_file)
        for task, mm in baseline_metrics.items():
            print(f'[학습 전 - {task}]'); display(pd.DataFrame(evaluate.summary_table(mm)))
    else:
        started = time.time()
        baseline_metrics, _ = run_eval(model, '학습 전')
        baseline_file.parent.mkdir(parents=True, exist_ok=True)
        baseline_file.write_text(json.dumps(baseline_metrics, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        print(f'저장: {baseline_file} ({(time.time() - started) / 60:.1f}분)')

## 7. 렌더링 확인 - 학습 샘플이 실제 입력과 같은지
`【 】` 안이 정답(loss 를 거는 부분)입니다. 확인할 것
- 챗봇: 시스템 프롬프트 → 도구 목록 → 사용자 발화 순서, 정답이 `<|tool_call>call:...{...}<tool_call|>` 형식
- 정답 끝이 `<turn|>`
- 사고(thinking) 지시가 들어가 있지 않음

학습·검증 샘플을 만드는 동안 `학습 샘플 - 챗봇  1200/3566 (33%) | 경과 20초 | 남은 시간 약 40초` 처럼 진행이 표시됩니다.

In [ ]:
# 반복·비율은 5번에서 이미 맞췄으므로 여기서는 반복하지 않습니다 (analysis_repeat=1, no_tool_repeat=1)
train_s, train_stats = samples.build_samples(tokenizer, *data['train'], reply_ratio=REPLY_RATIO)
val_s, val_stats = samples.build_samples(tokenizer, *data['val'], reply_ratio=REPLY_RATIO)
print('학습:', train_stats)
print('검증:', val_stats)
for task in ('chat_call', 'chat_reply', 'analysis'):
    s = next((x for x in train_s if x.task == task), None)
    if s is not None:
        print('\n' + '=' * 100 + '\n' + samples.show(tokenizer, s))

## 8. LoRA 붙이기 + 학습
**진행 표시** - transformers 가 표를 띄웁니다: `[ 40/636 12:05 < 3:00:10, Epoch 0.19/3]` = 전체 636 스텝 중 40 스텝, 경과 12분, 남은 시간 약 3시간.

**실시간 loss 그래프** - 표 아래 그래프가 매 스텝(`cfg.logging_steps`) 같은 자리에서 다시 그려집니다.
- 왼쪽 **train loss**: 옅은 선 = 스텝별 값(배치마다 들쭉날쭉함), 진한 선 = 이동 평균. 진한 선이 내려가면 학습이 되고 있는 것입니다.
- 가운데 **val loss (전체)**: 검증 때마다(`eval_steps`) 점이 하나씩 늘어납니다. ★ = 지금까지 가장 낮은 값 - 학습이 끝나면 이 체크포인트로 되돌아갑니다.
- 오른쪽 **과제별 val loss** (로그 눈금): 챗봇 도구 호출 / 챗봇 안내 문구 / 민원 분석을 따로 그립니다. 전체 val loss 는 쉬운 챗봇 샘플이 대부분이라
  분석 쪽이 나빠져도 가운데 그래프에는 잘 안 보입니다. **분석 선이 혼자 올라가기 시작하면** 반복해 넣은 분석 데이터를 외우기 시작한 것입니다.
  (검증 셋을 전체 + 과제별로 두 번 돌기 때문에 검증 시간이 약 2배입니다)
- train 은 계속 내려가는데 val 이 올라가기 시작하면 과적합입니다. (early stopping 이 3번 연속이면 자동으로 멈춤 - 기준은 전체 val loss)
- 학습이 끝나면 검증 때마다의 과제별 loss 표가 출력됩니다.
- 끝나면 `outputs/plots/<RUN_NAME>_loss.png` 로 저장됩니다.

In [ ]:
model, targets = trainer.attach_lora(model, cfg)
print('LoRA 위치:', trainer.summarize_targets(targets))
print(trainer.trainable_report(model))

live_plot = plots.LiveLossPlot(title=RUN_NAME, save_path=PLOT_DIR / f'{RUN_NAME}_loss.png')   # 스텝마다 갱신되는 loss 그래프
hf_trainer, plan = trainer.build_trainer(model, tokenizer, train_s, val_s, cfg, common.CHECKPOINT_DIR / RUN_NAME,
                                         extra_callbacks=[live_plot])
print('계획:', plan)
started = time.time()
hf_trainer.train()
model = trainer.finish_training(hf_trainer)
train_minutes = round((time.time() - started) / 60, 1)
history = trainer.loss_history(hf_trainer)
print('학습 시간(분):', train_minutes, '| 최저 검증 loss(전체):', hf_trainer.state.best_metric)
print('[검증 loss - 전체(all) / 과제별]  ★ 행 = 학습 끝에 되돌아간 체크포인트')
val_table = pd.DataFrame(trainer.val_loss_table(hf_trainer))
val_table['best'] = ['★' if s == hf_trainer.state.best_global_step else '' for s in val_table['step']] if getattr(hf_trainer.state, 'best_global_step', None) else ''
display(val_table)

## 9. 학습 후 평가 + 학습 전과 비교 (평가셋 - test 분할)
같은 지표를 나중에 PC 의 Ollama 로도 잽니다(README 참고). 두 결과가 크게 다르면 Ollama 의 프롬프트 조립이 학습 때와 다르다는 뜻입니다.
학습 전 평가(6-1)가 있으면 비교표 아래에 **학습 전/후 막대그래프**가 과제별로 그려집니다. 회색 = 학습 전, 파랑 = 학습 후, 오른쪽 숫자 = 변화(%p, 초록 = 좋아짐 / 빨강 = 나빠짐).
`outputs/plots/<RUN_NAME>_compare.png` 로 저장됩니다.

모드 1(문의)은 학습하지 않았으므로 답변이 이상하지 않은지 눈으로만 확인합니다.

In [ ]:
metrics, gen = run_eval(model, '학습 후')
if baseline_metrics:
    print('[학습 전 → 학습 후]')
    comparison = evaluate.compare_table(baseline_metrics, metrics)
    display(pd.DataFrame(comparison))
    plots.plot_comparison(comparison, save_path=PLOT_DIR / f'{RUN_NAME}_compare.png')
chat_m = metrics['chat']
print('[chat - 정답 종류 6종별 도구 선택 (맞음/전체)]  학습 전:', (baseline_metrics.get('chat') or {}).get('_종류별 도구 선택(맞음/전체)'),
      '→ 학습 후:', chat_m['_종류별 도구 선택(맞음/전체)'])
print('[chat - confusion (정답 -> 예측)]'); display(pd.DataFrame(list(chat_m['_confusion(정답, 예측)'].items()), columns=['정답 -> 예측', '건수']))
if 'analysis' in metrics:
    print('[analysis] 생성 잘림 건수:', metrics['analysis'].get('_생성 잘림 건수'))
for task, mm in metrics.items():
    if mm.get('_틀린 예'):
        print(f'[{task} - 틀린 예]'); display(pd.DataFrame(mm['_틀린 예'][:10]))
print('[모드 1 답변 - 학습하지 않은 모드가 망가지지 않았는지 눈으로 확인]')
display(pd.DataFrame(evaluate.mode1_answers(gen)))

## 10. 어댑터 저장 → `model-training/outputs/adapters/<RUN_NAME>/`

In [ ]:
run_dir = trainer.save_run(model, tokenizer, common.ADAPTERS_DIR / RUN_NAME, {
    'run_name': RUN_NAME, 'smoke': SMOKE, 'train_config': cfg.to_dict(), 'eval_version': evaluate.EVAL_VERSION, 'analysis_prompt_style': ANALYSIS_PROMPT_STYLE,
    'train_limit': TRAIN_LIMIT, 'max_repeat': MAX_REPEAT, 'analysis_repeat': ANALYSIS_REPEAT,
    'train_kinds': {'chat': records.kind_counts(data['train'][0], records.chat_kind),
                    'analysis': records.kind_counts(data['train'][1], records.analysis_kind)},
    'sample_stats': {'train': train_stats, 'val': val_stats}, 'train_minutes': train_minutes,
    'best_eval_loss': hf_trainer.state.best_metric,
    'val_loss_by_task': trainer.val_loss_table(hf_trainer),
    'metrics_before': _scores(baseline_metrics) if baseline_metrics else None,
    'metrics': _scores(metrics),
    'comparison': evaluate.compare_table(baseline_metrics, metrics) if baseline_metrics else None,
})
print('저장:', run_dir)

## 11. GGUF 내보내기 → PC 로 바로 다운로드
학습에 쓴 모델을 메모리에서 내리고, 베이스를 bf16(원본 정밀도)으로 **CPU 에** 다시 올려 어댑터를 합친 뒤 GGUF 로 바꿉니다.
병합 모델(약 16GB)과 GGUF(약 8GB)는 **드라이브를 거치지 않고** Colab 로컬 디스크(`/content/export`)에만 만들고, 끝나면 PC 로 바로 내려받습니다.
(드라이브 용량이 넉넉하면 `TO_DRIVE = True` 로 바꿔 `model-training/modelfiles/<RUN_NAME>/` 에 저장할 수도 있습니다.) 20~40분 걸립니다.

**진행 표시** - `[1/4] 어댑터 병합` → `[2/4] 변환 스크립트 준비` → `[3/4] GGUF 변환` → `[4/4] 복사/Modelfile` 순서로 단계와 경과 시간이 나옵니다.
3단계에서는 20초마다 `[GGUF 변환] 파일 쓰는 중 3.2GB / 약 8.7GB (약 37%) | 남은 시간 …` 이 출력되고, 끝나면 GGUF 가 온전한지(텐서 개수·크기) 검증합니다.

**다운로드** - 브라우저 기본 다운로드 폴더(Windows 는 보통 `C:\Users\<이름>\Downloads`)에 `.gguf` 와 `Modelfile` 이 저장됩니다.
크롬이 "여러 파일 다운로드 허용" 을 물으면 **허용**하세요. 다운로드가 끝나기 전에는 Colab 세션을 닫지 마세요.

In [ ]:
TO_DRIVE = False   # False: 드라이브에 쓰지 않고 Colab 로컬 디스크에만 만들어 PC 로 바로 다운로드 (드라이브 용량 절약)

for _name in ('hf_trainer', 'model', 'gen'):   # 학습에 쓴 모델을 GPU 에서 내림 (평가를 건너뛰었어도 동작)
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

info = export.export_all(run_dir, RUN_NAME, outtype=GGUF_OUTTYPE, work_dir='/content/export', to_drive=TO_DRIVE)
print(json.dumps(info, ensure_ascii=False, indent=2))

if TO_DRIVE:
    !ls -la {common.EXPORT_DIR / RUN_NAME}
else:
    # Modelfile + GGUF 를 PC 로 직접 다운로드 (브라우저 기본 다운로드 폴더에 저장됨)
    from google.colab import files
    from pathlib import Path
    out_files = sorted(Path('/content/export/out').glob('*'), key=lambda p: p.suffix == '.gguf')   # Modelfile 먼저
    print('다운로드 시작:', [f'{f.name} ({f.stat().st_size / 1e9:.2f}GB)' for f in out_files])
    for f in out_files:
        files.download(str(f))

## 11-B. llama-server 용 내보내기 (베이스 GGUF + 어댑터 GGUF) → PC 로 바로 다운로드
`LLM_BACKEND=llama_server` 로 쓸 때의 내보내기입니다. **병합하지 않고** 어댑터를 따로 GGUF 로 바꿉니다.
llama-server 가 베이스 모델 하나를 올려 두고, 요청마다 어댑터를 켜고 끕니다. (민원 분석·챗봇 = 켬, CSV 열 매핑 = 베이스)

| 파일 | 크기 | 언제 만드나 |
|---|---|---|
| `gemma-4-E4B-it-q8_0.gguf` (베이스) | 약 8GB | **처음 한 번만** (`MAKE_BASE_GGUF = True`). 같은 베이스로 학습하는 동안 계속 같이 씀 |
| `complaint-lora-<RUN_NAME>-f16.gguf` (어댑터) | 수십~수백 MB | 학습할 때마다 |

- 11번(Ollama 용)을 실행하지 않았어도 됩니다. 10번(어댑터 저장)까지만 끝나 있으면 됩니다.
- PC 에 베이스 GGUF 가 이미 있으면 `MAKE_BASE_GGUF = False` 로 두세요. 어댑터만 1~2분이면 끝납니다.
- 베이스 GGUF 는 Colab 디스크에 학습 베이스 원본(약 16GB)을 받아 변환하므로 15~30분 걸립니다. (끝나면 원본은 지움)
- 다운로드 위치·주의 사항은 11번과 같습니다.

In [ ]:
MAKE_BASE_GGUF = False    # 처음 한 번만 True. PC 의 model-training/modelfiles/llama-server/ 에 베이스 GGUF 가 이미 있으면 False
LS_TO_DRIVE = False      # False: Colab 로컬 디스크에만 만들어 PC 로 바로 다운로드 / True: 드라이브 model-training/modelfiles/llama-server/

for _name in ('hf_trainer', 'model', 'gen'):   # 학습에 쓴 모델을 GPU 에서 내림 (11번을 건너뛰었을 때)
    globals().pop(_name, None)
gc.collect(); torch.cuda.empty_cache()

ls_info = export.export_llama_server(run_dir, RUN_NAME, make_base=MAKE_BASE_GGUF, base_outtype=GGUF_OUTTYPE,
                                     work_dir='/content/export_ls', to_drive=LS_TO_DRIVE)
print(json.dumps(ls_info, ensure_ascii=False, indent=2))

if LS_TO_DRIVE:
    !ls -la {common.EXPORT_DIR / export.LLAMA_SERVER_DIR_NAME}
else:
    from google.colab import files
    from pathlib import Path
    out_files = sorted(Path('/content/export_ls/out').glob('*'), key=lambda p: p.stat().st_size)   # 작은 파일(안내·어댑터) 먼저
    print('다운로드 시작:', [f'{f.name} ({f.stat().st_size / 1e9:.2f}GB)' for f in out_files])
    for f in out_files:
        files.download(str(f))

## 12. PC 에서 등록·평가

### 12-A. Ollama (11번으로 만든 병합 모델)
1. 11번에서 내려받은 `.gguf` 와 `Modelfile` 을 **같은 폴더**에 둡니다. (`TO_DRIVE = True` 였다면 드라이브의 `model-training/modelfiles/<RUN_NAME>/` 에서 받음)
2. 그 폴더에서 등록합니다. (bf16 으로 만들었다면 끝에 `--quantize q4_K_M`)
   ```
   ollama create complaint-gemma4 -f Modelfile
   ```
3. 저장소 루트에서 Ollama 경로로 평가합니다. (dcm-back 가상환경)
   ```
   python model-training/evaluate.py --ollama complaint-gemma4:latest --limit 300
   ```
   9번 결과와 비슷하면 학습 때와 실행 때 프롬프트가 같게 조립되고 있는 것입니다.
4. 앱에 적용: `dcm-back/.env` 의 `LLM_BACKEND=ollama`, `LLM_MODEL=complaint-gemma4:latest`

`SMOKE = True` 로 돌렸다면 2~3번까지 해 보고(점수는 낮아도 정상), 문제가 없으면 4번 셀에서 `SMOKE = False` 로 본 학습을 하세요.

### 12-B. llama-server (11-B 로 만든 베이스 + 어댑터)
1. 내려받은 `.gguf` 두 개를 저장소의 `model-training/modelfiles/llama-server/` 에 둡니다. (`llama_server_info_<RUN_NAME>.json` 에 실행 명령이 있음)
2. [llama.cpp 릴리스](https://github.com/ggml-org/llama.cpp/releases)에서 Windows 빌드(NVIDIA 는 cuda)를 받아 `llama-server.exe` 위치를 확인합니다.
3. 저장소 루트에서 서버를 띄웁니다.
   ```
   .\scripts\run_llama_server.ps1 -LlamaServer C:\llama.cpp\llama-server.exe -Adapter .\model-training\modelfiles\llama-server\complaint-lora-<RUN_NAME>-f16.gguf
   ```
4. 같은 서버로 어댑터 켬/끔을 각각 평가합니다. 켬 점수가 9번 결과와 비슷하면 정상입니다.
   ```
   python model-training/evaluate.py --llama-server http://127.0.0.1:8080 --limit 300
   python model-training/evaluate.py --llama-server http://127.0.0.1:8080 --lora-scale 0 --limit 300
   ```
5. 앱에 적용: `dcm-back/.env` 의 `LLM_BACKEND=llama_server` (어댑터를 켤 작업은 `LLM_ADAPTER_TASKS=analysis,chat`)